In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

start_time = spark.sql(
    "SELECT current_timestamp()"
).first()[0]
crypto_schema = StructType([
    StructField("high", DoubleType(), True),
    StructField("open", DoubleType(), True),
    StructField("volume", DoubleType(), True),
    StructField("close", DoubleType(), True),
    StructField("low", DoubleType(), True),
    StructField("timestamp", LongType(), True),
    StructField("ticker", StringType(), True)
])

print("Schema Created Successfully")
# Created the schema

# Read csv file and define dataframe
df = spark.read \
    .schema(crypto_schema) \
    .option("header", "true") \
    .csv("/Volumes/workspace/default/crypto_data/")
df.count()

# Add the timestamp column
bronze_df = df.withColumn(
    "load_timestamp",
    current_timestamp()
)

bronze_df.count()

# Store processed Bronze data as a Delta table
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_crypto_market")

# Check that data was loaded successfully into the Bronze table
spark.sql("""
SELECT *
FROM bronze_crypto_market
LIMIT 5
""").show()

silver_df = bronze_df.withColumn(
    "observation_date",
    to_date(from_unixtime(col("timestamp")))
)
silver_df.select(
    "ticker",
    "timestamp",
    "observation_date"
).show(10, False)

Schema Created Successfully


In [0]:
from pyspark.sql.functions import *

# Split ticker into coin symbol and quote currency for easier analysis

silver_df = silver_df \
    .withColumn(
        "coin_symbol",
        split(col("ticker"), "-")[0]
    ) \
    .withColumn(
        "quote_currency",
        split(col("ticker"), "-")[1]
    )

silver_df.select(
    "ticker",
    "coin_symbol",
    "quote_currency"
).show(10, False)


+------------+-----------+--------------+
|ticker      |coin_symbol|quote_currency|
+------------+-----------+--------------+
|SAFEBULL-USD|SAFEBULL   |USD           |
|SAFEBULL-USD|SAFEBULL   |USD           |
|SAFEBULL-USD|SAFEBULL   |USD           |
|SAFEBULL-USD|SAFEBULL   |USD           |
|SAFEBULL-USD|SAFEBULL   |USD           |
|SAFEBULL-USD|SAFEBULL   |USD           |
|SAFEBULL-USD|SAFEBULL   |USD           |
|SAFEBULL-USD|SAFEBULL   |USD           |
|SAFEBULL-USD|SAFEBULL   |USD           |
|SAFEBULL-USD|SAFEBULL   |USD           |
+------------+-----------+--------------+
only showing top 10 rows


In [0]:
# Select required columns and rename price fields for a clean Silver layer schema
silver_df = silver_df.select(
    "coin_symbol",
    "quote_currency",
    "observation_date",
    col("open").alias("open_price"),
    col("high").alias("high_price"),
    col("low").alias("low_price"),
    col("close").alias("close_price"),
    "volume",
    "load_timestamp"
).withColumn(
    "market_cap",
    lit(None).cast("double")
)
silver_df.printSchema()

root
 |-- coin_symbol: string (nullable = true)
 |-- quote_currency: string (nullable = true)
 |-- observation_date: date (nullable = true)
 |-- open_price: double (nullable = true)
 |-- high_price: double (nullable = true)
 |-- low_price: double (nullable = true)
 |-- close_price: double (nullable = true)
 |-- volume: double (nullable = true)
 |-- load_timestamp: timestamp (nullable = false)
 |-- market_cap: double (nullable = true)



In [0]:
# Remove duplicate records based on coin and observation date
silver_df = silver_df.dropDuplicates(
    ["coin_symbol", "observation_date"]
)
silver_df.count()

8114768

In [0]:
silver_df.write \
    .format("delta") \
    .option("mergeSchema", "true") \
    .mode("overwrite") \
    .saveAsTable("silver_crypto_market")

spark.sql("""
SELECT *
FROM silver_crypto_market
LIMIT 5
""").show()




+-----------+--------------+----------------+-----------------+----------+-----------------+-----------------+------------------+--------------------+----------+
|coin_symbol|quote_currency|observation_date|       open_price|high_price|        low_price|      close_price|            volume|      load_timestamp|market_cap|
+-----------+--------------+----------------+-----------------+----------+-----------------+-----------------+------------------+--------------------+----------+
|   SAFEBULL|           USD|      2022-05-11|              0.0|    3583.0|              0.0|              0.0|               0.0|2026-10-10 19:46:...|      NULL|
|   SAFEBULL|           USD|      2025-05-11|              0.0|       0.0|              0.0|              0.0|               0.0|2026-10-10 19:46:...|      NULL|
|  SATO36386|           USD|      2026-04-29|              0.0|       0.0|              0.0|              0.0|               0.0|2026-10-10 19:46:...|      NULL|
|        SDM|           USD|

In [0]:
# Read CoinGecko incremental JSON file
incremental_file = "markets(2).json"
incremental_df = spark.read.json(
    f"/Volumes/workspace/default/crypto_data/{incremental_file}"
)

# Display sample records
incremental_df.show(5)

# Display schema
incremental_df.printSchema()

# Count records
incremental_df.count()

+--------+---------------------+--------------------+----------+---------------------+--------------------+--------------------+-------------+-----------------------+--------+-----------+--------------------+--------------------+--------+-------------+---------------------+--------------------------------+---------------+----------+--------+-------------------+---------------------------+--------------------+------+--------------------+---------------+
|     ath|ath_change_percentage|            ath_date|       atl|atl_change_percentage|            atl_date|  circulating_supply|current_price|fully_diluted_valuation|high_24h|         id|               image|        last_updated| low_24h|   market_cap|market_cap_change_24h|market_cap_change_percentage_24h|market_cap_rank|max_supply|    name|   price_change_24h|price_change_percentage_24h|                 roi|symbol|        total_supply|   total_volume|
+--------+---------------------+--------------------+----------+---------------------+

250

In [0]:
from pyspark.sql.functions import *
incremental_silver_df = incremental_df.select(
    upper(col("symbol")).alias("coin_symbol"),

    lit("USD").alias("quote_currency"),

    to_date(col("last_updated")).alias("observation_date"),

    lit(None).cast("double").alias("open_price"),

    col("high_24h").cast("double").alias("high_price"),

    col("low_24h").cast("double").alias("low_price"),

    col("current_price").cast("double").alias("close_price"),

    col("total_volume").cast("double").alias("volume"),

    current_timestamp().alias("load_timestamp"),

    col("market_cap").cast("double").alias("market_cap")
)

In [0]:
incremental_silver_df.printSchema()
incremental_silver_df.show(5)
print("Rows:", incremental_df.count())

root
 |-- coin_symbol: string (nullable = true)
 |-- quote_currency: string (nullable = false)
 |-- observation_date: date (nullable = true)
 |-- open_price: double (nullable = true)
 |-- high_price: double (nullable = true)
 |-- low_price: double (nullable = true)
 |-- close_price: double (nullable = true)
 |-- volume: double (nullable = true)
 |-- load_timestamp: timestamp (nullable = false)
 |-- market_cap: double (nullable = true)

+-----------+--------------+----------------+----------+----------+---------+-----------+---------------+--------------------+-----------------+
|coin_symbol|quote_currency|observation_date|open_price|high_price|low_price|close_price|         volume|      load_timestamp|       market_cap|
+-----------+--------------+----------------+----------+----------+---------+-----------+---------------+--------------------+-----------------+
|        BTC|           USD|      2026-10-10|      NULL|   83094.0|  82262.0|    82961.0|1.6109975008E10|2026-10-10 19:46:...

In [0]:
incremental_silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("incremental_silver_market")

spark.sql("""
SELECT *
FROM incremental_silver_market
LIMIT 5
""").show()

+-----------+--------------+----------------+----------+----------+---------+-----------+---------------+--------------------+-----------------+
|coin_symbol|quote_currency|observation_date|open_price|high_price|low_price|close_price|         volume|      load_timestamp|       market_cap|
+-----------+--------------+----------------+----------+----------+---------+-----------+---------------+--------------------+-----------------+
|        BTC|           USD|      2026-10-10|      NULL|   83094.0|  82262.0|    82961.0|1.6109975008E10|2026-10-10 19:46:...|1.667205205117E12|
|        ETH|           USD|      2026-10-10|      NULL|   2513.03|  2474.44|    2510.77|  5.409131403E9|2026-10-10 19:46:...| 3.06620266049E11|
|       USDT|           USD|      2026-10-10|      NULL|  0.999347| 0.999124|    0.99917| 3.086203076E10|2026-10-10 19:46:...| 1.84099956114E11|
|        BNB|           USD|      2026-10-10|      NULL|     752.9|   737.32|     749.99|   4.94166138E8|2026-10-10 19:46:...|  9.

In [0]:
spark.sql("""

MERGE INTO silver_crypto_market AS target

USING incremental_silver_market AS source

ON target.coin_symbol = source.coin_symbol
AND target.observation_date = source.observation_date

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *

""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
spark.sql("""
SELECT *
FROM silver_crypto_market
WHERE observation_date = '2026-10-10'
LIMIT 20
""").show()

spark.sql("""
SELECT COUNT(*)
FROM silver_crypto_market
""").show()

+-----------+--------------+----------------+----------+----------+----------+-----------+---------------+--------------------+-----------------+
|coin_symbol|quote_currency|observation_date|open_price|high_price| low_price|close_price|         volume|      load_timestamp|       market_cap|
+-----------+--------------+----------------+----------+----------+----------+-----------+---------------+--------------------+-----------------+
|        BTC|           USD|      2026-10-10|      NULL|   83094.0|   82262.0|    82961.0|1.6109975008E10|2026-10-10 19:46:...|1.667205205117E12|
|        ETH|           USD|      2026-10-10|      NULL|   2513.03|   2474.44|    2510.77|  5.409131403E9|2026-10-10 19:46:...| 3.06620266049E11|
|       USDT|           USD|      2026-10-10|      NULL|  0.999347|  0.999124|    0.99917| 3.086203076E10|2026-10-10 19:46:...| 1.84099956114E11|
|        BNB|           USD|      2026-10-10|      NULL|     752.9|    737.32|     749.99|   4.94166138E8|2026-10-10 19:46:.

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import current_timestamp
import time

rows_processed = incremental_silver_df.count()
print(rows_processed)
run_id = int(time.time())
end_time = spark.sql(
    "SELECT current_timestamp()"
).first()[0]
log_data = [(
    run_id,
    "Incremental Merge",
    incremental_file,
    "SUCCESS",
    rows_processed,
    start_time,
    end_time
)]

log_df = spark.createDataFrame(
    log_data,
    [
        "run_id",
        "layer",
        "file_processed",
        "status",
        "rows_processed",
        "start_time",
        "end_time"
    ]
)

log_df = log_df.withColumn(
    "execution_time",
    current_timestamp()
)

250


In [0]:
log_df.write \
    .format("delta") \
    .option("mergeSchema", "true") \
    .mode("append") \
    .saveAsTable("pipeline_execution_logs")

In [0]:
spark.sql("""
SELECT *
FROM pipeline_execution_logs
""").show(truncate=False)

+----------+-----------------+---------------+-------+--------------+--------------------------+--------------------------+--------------------------+
|run_id    |layer            |file_processed |status |rows_processed|execution_time            |start_time                |end_time                  |
+----------+-----------------+---------------+-------+--------------+--------------------------+--------------------------+--------------------------+
|1791661601|Incremental Merge|markets(2).json|SUCCESS|250           |2026-10-10 19:47:24.99452 |2026-10-10 19:45:56.890369|2026-10-10 19:46:41.706223|
|1791661034|Incremental Merge|markets(2).json|SUCCESS|250           |2026-10-10 19:37:17.987634|NULL                      |NULL                      |
|1791657963|Incremental Merge|markets.json   |SUCCESS|250           |2026-10-10 18:46:04.808516|NULL                      |NULL                      |
|1791660990|Incremental Merge|markets.json   |SUCCESS|250           |2026-10-10 19:36:35.87819

In [0]:
# Gold Layer imports and execution start time

from pyspark.sql.functions import *
from pyspark.sql.window import Window
import time

gold_start_time = spark.sql(
    "SELECT current_timestamp()"
).first()[0]

print("Gold layer processing started at:", gold_start_time)

Gold layer processing started at: 2026-10-25 18:27:15.125237


In [0]:
# Read the final Silver table containing historical and incremental data

silver_gold_source_df = spark.table("silver_crypto_market")

print("Silver rows available for Gold processing:")
print(silver_gold_source_df.count())

silver_gold_source_df.printSchema()

Silver rows available for Gold processing:
8115018
root
 |-- coin_symbol: string (nullable = true)
 |-- quote_currency: string (nullable = true)
 |-- observation_date: date (nullable = true)
 |-- open_price: double (nullable = true)
 |-- high_price: double (nullable = true)
 |-- low_price: double (nullable = true)
 |-- close_price: double (nullable = true)
 |-- volume: double (nullable = true)
 |-- load_timestamp: timestamp (nullable = true)
 |-- market_cap: double (nullable = true)



In [0]:
# Create the Coin dimension from distinct coin and quote-currency combinations

dim_coin_df = (
    silver_gold_source_df
    .select(
        upper(trim(col("coin_symbol"))).alias("coin_symbol"),
        upper(trim(col("quote_currency"))).alias("quote_currency")
    )
    .filter(
        col("coin_symbol").isNotNull() &
        col("quote_currency").isNotNull()
    )
    .dropDuplicates(
        ["coin_symbol", "quote_currency"]
    )
    .withColumn(
        "coin_key",
        xxhash64(
            col("coin_symbol"),
            col("quote_currency")
        )
    )
    .select(
        "coin_key",
        "coin_symbol",
        "quote_currency"
    )
)

dim_coin_df.show(10, truncate=False)

print("Coin dimension rows:", dim_coin_df.count())

+--------------------+------------+--------------+
|coin_key            |coin_symbol |quote_currency|
+--------------------+------------+--------------+
|3861185676359342152 |SOLALA      |USD           |
|6060196218719592513 |SPACEPI28304|USD           |
|7803303027090560192 |STUSDT      |USD           |
|186110040613145060  |SOUL        |USD           |
|-6818400055145542492|STIMA       |USD           |
|-3402542173140089734|SLICE       |USD           |
|3304578021358712740 |SLND        |USD           |
|-688321982761655751 |SDEX        |USD           |
|2707095143758718511 |STACCANA    |USD           |
|-3196046461939023269|SMH39498    |USD           |
+--------------------+------------+--------------+
only showing top 10 rows
Coin dimension rows: 8288


In [0]:
# Save the Coin dimension as a Gold Delta table

dim_coin_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_coin")

In [0]:
# Create the Date dimension from distinct Silver observation dates

dim_date_df = (
    silver_gold_source_df
    .select(
        col("observation_date").alias("full_date")
    )
    .filter(
        col("full_date").isNotNull()
    )
    .dropDuplicates(
        ["full_date"]
    )
    .withColumn(
        "date_key",
        date_format(col("full_date"), "yyyyMMdd").cast("int")
    )
    .withColumn(
        "day",
        dayofmonth(col("full_date"))
    )
    .withColumn(
        "month",
        month(col("full_date"))
    )
    .withColumn(
        "month_name",
        date_format(col("full_date"), "MMMM")
    )
    .withColumn(
        "quarter",
        quarter(col("full_date"))
    )
    .withColumn(
        "year",
        year(col("full_date"))
    )
    .withColumn(
        "weekday",
        date_format(col("full_date"), "EEEE")
    )
    .select(
        "date_key",
        "full_date",
        "day",
        "month",
        "month_name",
        "quarter",
        "year",
        "weekday"
    )
)

dim_date_df.show(10, truncate=False)

print("Date dimension rows:", dim_date_df.count())

+--------+----------+---+-----+----------+-------+----+---------+
|date_key|full_date |day|month|month_name|quarter|year|weekday  |
+--------+----------+---+-----+----------+-------+----+---------+
|20160622|2016-06-22|22 |6    |June      |2      |2016|Wednesday|
|20181114|2018-11-14|14 |11   |November  |4      |2018|Wednesday|
|20220922|2022-09-22|22 |9    |September |3      |2022|Thursday |
|20141011|2014-10-11|11 |10   |October   |4      |2014|Saturday |
|20230115|2023-01-15|15 |1    |January   |1      |2023|Sunday   |
|20230226|2023-02-26|26 |2    |February  |1      |2023|Sunday   |
|20200416|2020-04-16|16 |4    |April     |2      |2020|Thursday |
|20200125|2020-01-25|25 |1    |January   |1      |2020|Saturday |
|20250625|2025-06-25|25 |6    |June      |2      |2025|Wednesday|
|20221104|2022-11-04|4  |11   |November  |4      |2022|Friday   |
+--------+----------+---+-----+----------+-------+----+---------+
only showing top 10 rows
Date dimension rows: 4407


In [0]:
# Save the Date dimension as a Gold Delta table

dim_date_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("dim_date")

In [0]:
# Prepare one cleaned Silver record per coin and observation date

fact_source_df = (
    silver_gold_source_df
    .select(
        upper(trim(col("coin_symbol"))).alias("coin_symbol"),
        upper(trim(col("quote_currency"))).alias("quote_currency"),
        col("observation_date"),
        col("open_price").cast("double"),
        col("high_price").cast("double"),
        col("low_price").cast("double"),
        col("close_price").cast("double"),
        col("volume").cast("double"),
        col("market_cap").cast("double"),
        col("load_timestamp")
    )
    .filter(
        col("coin_symbol").isNotNull() &
        col("quote_currency").isNotNull() &
        col("observation_date").isNotNull()
    )
    .dropDuplicates(
        [
            "coin_symbol",
            "quote_currency",
            "observation_date"
        ]
    )
)

In [0]:
# Define the window used to find the previous available daily price

coin_date_window = (
    Window
    .partitionBy(
        "coin_symbol",
        "quote_currency"
    )
    .orderBy(
        "observation_date"
    )
)

fact_with_return_df = (
    fact_source_df
    .withColumn(
        "previous_close_price",
        lag("close_price", 1).over(coin_date_window)
    )
    .withColumn(
        "daily_return_pct",
        when(
            col("previous_close_price").isNull() |
            (col("previous_close_price") == 0),
            lit(None).cast("double")
        ).otherwise(
            (
                (
                    col("close_price") -
                    col("previous_close_price")
                ) /
                col("previous_close_price")
            ) * 100
        )
    )
)

In [0]:
# Replace descriptive Coin and Date values with dimension foreign keys

fact_daily_market_df = (
    fact_with_return_df
    .withColumn(
        "coin_key",
        xxhash64(
            col("coin_symbol"),
            col("quote_currency")
        )
    )
    .withColumn(
        "date_key",
        date_format(
            col("observation_date"),
            "yyyyMMdd"
        ).cast("int")
    )
    .select(
        "coin_key",
        "date_key",
        "open_price",
        "high_price",
        "low_price",
        "close_price",
        "volume",
        "market_cap",
        "daily_return_pct",
        "load_timestamp"
    )
)

fact_daily_market_df.show(10, truncate=False)

print("Fact table rows:", fact_daily_market_df.count())

+--------------------+--------+-----------+------------------+------------------+------------------+------------------+----------+-------------------+--------------------------+
|coin_key            |date_key|open_price |high_price        |low_price         |close_price       |volume            |market_cap|daily_return_pct   |load_timestamp            |
+--------------------+--------+-----------+------------------+------------------+------------------+------------------+----------+-------------------+--------------------------+
|-3864210127344666816|20221011|8680395.0  |0.919854998588562 |1.4891279935836792|1.1901439428329468|1.6600420475006104|NULL      |NULL               |2026-10-10 19:46:20.396687|
|-3864210127344666816|20221012|1.0585178E7|0.9015570282936096|0.9502350091934204|1.488327980041504 |1.5013539791107178|NULL      |25.05445152279462  |2026-10-10 19:46:20.396687|
|-3864210127344666816|20221013|5019056.0  |0.623458981513977 |0.7151749730110168|0.9578440189361572|1.02836203

In [0]:
# Save the daily market fact data as a Gold Delta table

fact_daily_market_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("fact_daily_market")

In [0]:
# GOLD VALIDATION, LOGGING, AND FINAL VERIFICATION

from pyspark.sql.functions import current_timestamp
import time


# 1. Validate the star-schema joins

gold_validation_df = spark.sql("""
SELECT
    c.coin_symbol,
    c.quote_currency,
    d.full_date,
    f.open_price,
    f.high_price,
    f.low_price,
    f.close_price,
    f.volume,
    f.market_cap,
    f.daily_return_pct
FROM fact_daily_market f
INNER JOIN dim_coin c
    ON f.coin_key = c.coin_key
INNER JOIN dim_date d
    ON f.date_key = d.date_key
LIMIT 20
""")

print("Sample Gold records:")
gold_validation_df.show(20, truncate=False)


# 2. Check for invalid Coin foreign keys

missing_coin_keys = spark.sql("""
SELECT COUNT(*) AS missing_coin_keys
FROM fact_daily_market f
LEFT JOIN dim_coin c
    ON f.coin_key = c.coin_key
WHERE c.coin_key IS NULL
""").first()[0]


# 3. Check for invalid Date foreign keys

missing_date_keys = spark.sql("""
SELECT COUNT(*) AS missing_date_keys
FROM fact_daily_market f
LEFT JOIN dim_date d
    ON f.date_key = d.date_key
WHERE d.date_key IS NULL
""").first()[0]

print("Missing coin keys:", missing_coin_keys)
print("Missing date keys:", missing_date_keys)


# 4. Stop the pipeline if foreign-key validation fails

if missing_coin_keys != 0 or missing_date_keys != 0:
    raise ValueError(
        "Gold validation failed because unmatched foreign keys were found."
    )


# 5. Capture Gold processing completion information

gold_end_time = spark.sql(
    "SELECT current_timestamp()"
).first()[0]

gold_rows_processed = fact_daily_market_df.count()
gold_run_id = int(time.time())


# 6. Create the Silver-to-Gold execution log

gold_log_data = [(
    gold_run_id,
    "Silver-to-Gold",
    "silver_crypto_market",
    "SUCCESS",
    gold_rows_processed,
    gold_start_time,
    gold_end_time
)]

gold_log_df = spark.createDataFrame(
    gold_log_data,
    [
        "run_id",
        "layer",
        "file_processed",
        "status",
        "rows_processed",
        "start_time",
        "end_time"
    ]
).withColumn(
    "execution_time",
    current_timestamp()
)


# 7. Append the Gold execution log

gold_log_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("pipeline_execution_logs")


# 8. Display final Gold table counts

print("Final Gold table counts:")

spark.sql("""
SELECT 'dim_coin' AS table_name, COUNT(*) AS row_count
FROM dim_coin

UNION ALL

SELECT 'dim_date' AS table_name, COUNT(*) AS row_count
FROM dim_date

UNION ALL

SELECT 'fact_daily_market' AS table_name, COUNT(*) AS row_count
FROM fact_daily_market
""").show(truncate=False)


# 9. Display the latest pipeline logs

print("Latest pipeline execution logs:")

spark.sql("""
SELECT *
FROM pipeline_execution_logs
ORDER BY execution_time DESC
LIMIT 10
""").show(truncate=False)


# 10. Final status

print("Gold layer completed successfully.")
print("Gold rows processed:", gold_rows_processed)
print("Gold start time:", gold_start_time)
print("Gold end time:", gold_end_time)

Sample Gold records:
+-----------+--------------+----------+----------+------------------+------------------+------------------+------------------+----------+---------------------+
|coin_symbol|quote_currency|full_date |open_price|high_price        |low_price         |close_price       |volume            |market_cap|daily_return_pct     |
+-----------+--------------+----------+----------+------------------+------------------+------------------+------------------+----------+---------------------+
|005930     |USD           |2026-02-12|0.0       |176762.109375     |178204.359375     |178821.9375       |182482.875        |NULL      |NULL                 |
|005930     |USD           |2026-02-13|0.0       |178204.359375     |181264.1875       |178204.359375     |184780.5          |NULL      |-0.3453592627582396  |
|005930     |USD           |2026-02-14|0.0       |181082.34375      |181502.625        |181264.1875       |181641.421875     |NULL      |1.7170332621106792   |
|005930     |USD   